## Notebook 1: Data Exploration

**Goal:** Explore the Fitbit dataset to understand its structure, quality, 
and key variables before building our analysis pipeline.

**Datasets used:**
- `dailyActivity_merged.csv` — steps, calories, active minutes per user per day
- `heartrate_seconds_merged.csv` — raw heart rate readings
- `minuteSleep_merged.csv` — minute-level sleep data
- `weightLogInfo_merged.csv` — weight and BMI logs

## 1. Environment Setup
Setting up Java environment and initializing the Spark session.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

# Start Spark session
spark = SparkSession.builder \
    .appName("WearableHealthAnalyzer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") 

print("Spark version:", spark.version)

Spark version: 4.1.1


## 2. Load Data

Loading all four datasets from both time periods into Spark DataFrames.

In [ ]:
BASE_PATH_1 = "../data/archive (3)/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/"
BASE_PATH_2 = "../data/archive (3)/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/"

def load_csv(filename):
    """Load a CSV from both time periods and combine into one DataFrame."""
    df1 = spark.read.csv(BASE_PATH_1 + filename, header=True, inferSchema=True)
    df2 = spark.read.csv(BASE_PATH_2 + filename, header=True, inferSchema=True)
    return df1.union(df2)

# Load all four datasets
daily_activity = load_csv("dailyActivity_merged.csv")
heart_rate     = load_csv("heartrate_seconds_merged.csv")
sleep          = load_csv("minuteSleep_merged.csv")
weight         = load_csv("weightLogInfo_merged.csv")

## 3. Data Exploration

### 3.1 Daily Activity

In [7]:
print("Shape:", (daily_activity.count(), len(daily_activity.columns)))
print("Unique users:", daily_activity.select("Id").distinct().count())
print("\nSchema:")
daily_activity.printSchema()

Shape: (1397, 15)
Unique users: 35

Schema:
root
 |-- Id: long (nullable = true)
 |-- ActivityDate: string (nullable = true)
 |-- TotalSteps: integer (nullable = true)
 |-- TotalDistance: double (nullable = true)
 |-- TrackerDistance: double (nullable = true)
 |-- LoggedActivitiesDistance: double (nullable = true)
 |-- VeryActiveDistance: double (nullable = true)
 |-- ModeratelyActiveDistance: double (nullable = true)
 |-- LightActiveDistance: double (nullable = true)
 |-- SedentaryActiveDistance: double (nullable = true)
 |-- VeryActiveMinutes: integer (nullable = true)
 |-- FairlyActiveMinutes: integer (nullable = true)
 |-- LightlyActiveMinutes: integer (nullable = true)
 |-- SedentaryMinutes: integer (nullable = true)
 |-- Calories: integer (nullable = true)



In [8]:
print("Sample rows:")
daily_activity.show(5)

# Summary statistics for key columsn
print("Summary statistics:")
daily_activity.select(
    "TotalSteps",
    "Calories",
    "VeryActiveMinutes",
    "SedentaryMinutes"
).describe().show()

Sample rows:
+----------+------------+----------+----------------+----------------+------------------------+------------------+------------------------+-------------------+-----------------------+-----------------+-------------------+--------------------+----------------+--------+
|        Id|ActivityDate|TotalSteps|   TotalDistance| TrackerDistance|LoggedActivitiesDistance|VeryActiveDistance|ModeratelyActiveDistance|LightActiveDistance|SedentaryActiveDistance|VeryActiveMinutes|FairlyActiveMinutes|LightlyActiveMinutes|SedentaryMinutes|Calories|
+----------+------------+----------+----------------+----------------+------------------------+------------------+------------------------+-------------------+-----------------------+-----------------+-------------------+--------------------+----------------+--------+
|1503960366|   3/25/2016|     11004| 7.1100001335144| 7.1100001335144|                     0.0|   2.5699999332428|        0.46000000834465|   4.07000017166138|                    0

### 3.2 Heart Rate, Sleep & Weight

In [9]:
print("Heart Rate rows:", heart_rate.count(), "| Unique users:", heart_rate.select("Id").distinct().count())
heart_rate.show(3)

print("Sleep rows:", sleep.count(), "| Unique users:", sleep.select("Id").distinct().count())
sleep.show(3)

print("Weight rows:", weight.count(), "| Unique users:", weight.select("Id").distinct().count())
weight.show(3)

Heart Rate rows: 3638339 | Unique users: 15
+----------+-------------------+-----+
|        Id|               Time|Value|
+----------+-------------------+-----+
|2022484408|4/1/2016 7:54:00 AM|   93|
|2022484408|4/1/2016 7:54:05 AM|   91|
|2022484408|4/1/2016 7:54:10 AM|   96|
+----------+-------------------+-----+
only showing top 3 rows
Sleep rows: 387080 | Unique users: 25
+----------+--------------------+-----+-----------+
|        Id|                date|value|      logId|
+----------+--------------------+-----+-----------+
|1503960366|3/13/2016 2:39:30 AM|    1|11114919637|
|1503960366|3/13/2016 2:40:30 AM|    1|11114919637|
|1503960366|3/13/2016 2:41:30 AM|    1|11114919637|
+----------+--------------------+-----+-----------+
only showing top 3 rows
Weight rows: 100 | Unique users: 13
+----------+--------------------+----------------+----------------+----+----------------+--------------+-------------+
|        Id|                Date|        WeightKg|    WeightPounds| Fat|      

### 3.3 Dataset Summary

- not all users tracked all metrics, analysis will work on users with complete data
- daily activity is our most complete dataset(35 users)
- weight data is too sparse for any strong conclusions

## 4. Data Cleaning

### 4.1 Remove days where device wasn't worn
Days with 0 steps AND 0 calories 

In [11]:
# Check how many zero-step days exist
zero_days = daily_activity.filter(
    (F.col("TotalSteps") == 0) & (F.col("Calories") == 0)
)
print("Non-wear days to remove:", zero_days.count())

# Remove non-wear days
daily_activity_clean = daily_activity.filter(
    ~((F.col("TotalSteps") == 0) & (F.col("Calories") == 0))
)

print("Rows before cleaning:", daily_activity.count())
print("Rows after cleaning:", daily_activity_clean.count())

Non-wear days to remove: 9
Rows before cleaning: 1397
Rows after cleaning: 1388


### 4.2 Fix date formats
Convert date/time columns from strings to proper timestamp types.

In [12]:
# convert ActivityDate from string to date
daily_activity_clean = daily_activity_clean.withColumn(
    "ActivityDate", F.to_date("ActivityDate", "M/dd/yyyy")
)

# convert hear rate time column to timestamp
heart_rate_clean = heart_rate.withColumn(
    "Time", F.to_timestamp(F.col("Time"), "M/d/yyyy h:mm:ss a")
)
# convert sleep time column to timestamp
sleep_clean = sleep.withColumn(
    "StartTime", F.to_timestamp(F.col("date"), "M/d/yyyy h:mm:ss a")
)

daily_activity_clean.select("Id", "ActivityDate").show(3)


+----------+------------+
|        Id|ActivityDate|
+----------+------------+
|1503960366|  2016-03-25|
|1503960366|  2016-03-26|
|1503960366|  2016-03-27|
+----------+------------+
only showing top 3 rows


### 4.3 Aggregate heart rate from seconds to daily averages

In [13]:
# extract date from timestamp, then aggregate to daily average
heart_rate_daily = heart_rate_clean \
    .withColumn("ActivityDate", F.to_date(F.col("Time"))) \
    .groupBy("Id", "ActivityDate") \
    .agg(
        F.round(F.avg("Value"), 1).alias("AvgHeartRate"),
        F.min("Value").alias("MinHeartRate"),
        F.max("Value").alias("MaxHeartRate")
    ) \
    .orderBy("Id", "ActivityDate")

print("Heart rate aggregated rows:", heart_rate_daily.count())
heart_rate_daily.show(5)

Heart rate aggregated rows: 469


+----------+------------+------------+------------+------------+
|        Id|ActivityDate|AvgHeartRate|MinHeartRate|MaxHeartRate|
+----------+------------+------------+------------+------------+
|2022484408|  2016-04-01|        88.6|          53|         182|
|2022484408|  2016-04-02|        72.1|          49|         103|
|2022484408|  2016-04-03|        74.4|          49|         126|
|2022484408|  2016-04-04|        78.3|          54|         150|
|2022484408|  2016-04-05|        83.5|          53|         143|
+----------+------------+------------+------------+------------+
only showing top 5 rows


### 4.4 Aggregate sleep from minutes to daily totals


In [16]:
# Re-clean sleep with correct timestamp parsing
sleep_clean = sleep.withColumn(
    "date", F.to_timestamp(F.col("date"), "M/d/yyyy h:mm:ss a")
)

# Aggregate sleep to daily totals
sleep_daily = sleep_clean \
    .withColumn("ActivityDate", F.to_date(F.col("date"))) \
    .filter(F.col("ActivityDate").isNotNull()) \
    .groupBy("Id", "ActivityDate") \
    .agg(
        F.count("value").alias("TotalSleepMinutes"),
        F.sum(F.when(F.col("value") == 1, 1).otherwise(0)).alias("AsleepMinutes"),
        F.sum(F.when(F.col("value") == 2, 1).otherwise(0)).alias("RestlessMinutes"),
        F.sum(F.when(F.col("value") == 3, 1).otherwise(0)).alias("AwakeMinutes")
    ) \
    .orderBy("Id", "ActivityDate")

print("Sleep aggregated rows:", sleep_daily.count())
sleep_daily.show(5)

Sleep aggregated rows: 901
+----------+------------+-----------------+-------------+---------------+------------+
|        Id|ActivityDate|TotalSleepMinutes|AsleepMinutes|RestlessMinutes|AwakeMinutes|
+----------+------------+-----------------+-------------+---------------+------------+
|1503960366|  2016-03-13|              426|          411|             15|           0|
|1503960366|  2016-03-14|              386|          354|             27|           5|
|1503960366|  2016-03-15|              335|          312|             16|           7|
|1503960366|  2016-03-16|              366|          333|             28|           5|
|1503960366|  2016-03-17|              437|          402|             34|           1|
+----------+------------+-----------------+-------------+---------------+------------+
only showing top 5 rows


In [15]:
sleep_clean.select("date").show(5)

+--------------------+
|                date|
+--------------------+
|3/13/2016 2:39:30 AM|
|3/13/2016 2:40:30 AM|
|3/13/2016 2:41:30 AM|
|3/13/2016 2:42:30 AM|
|3/13/2016 2:43:30 AM|
+--------------------+
only showing top 5 rows
